In [ ]:
# Exploratory Data Analysis Lab Assignments - Python / Google Colab
# Converted from the uploaded R lab document.
# Run this file in Google Colab. It downloads the same CreditCard.csv dataset
# and loads the AirPassengers and mtcars datasets corresponding to the R datasets.

import sys
import subprocess
import importlib.util

def install_if_missing(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for package, import_name in [
    ("pandas", "pandas"),
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("statsmodels", "statsmodels"),
    ("plotly", "plotly"),
]:
    install_if_missing(package, import_name)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.datasets import get_rdataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("=" * 75)
print("EXPLORATORY DATA ANALYSIS LAB ASSIGNMENTS")
print("Python conversion for Google Colab")
print("=" * 75)

# ---------------------------------------------------------------------------
# DATASET DOWNLOADS
# ---------------------------------------------------------------------------

CREDITCARD_URL = "https://raw.githubusercontent.com/salemprakash/EDA/main/Data/CreditCard.csv"

print("\nDownloading the same CreditCard.csv dataset used in the R assignments...")
df = pd.read_csv(CREDITCARD_URL)
print("CreditCard dataset loaded:", df.shape)
print(df.head())

# Load the R built-in datasets used in the document.
print("\nLoading the AirPassengers dataset...")
air_data = get_rdataset("AirPassengers", "datasets").data
air_values = air_data.iloc[:, 0].astype(float)
air_index = pd.date_range(start="1949-01-01", periods=len(air_values), freq="MS")
air_passengers = pd.Series(air_values.values, index=air_index, name="Passengers")

print("\nLoading the mtcars dataset...")
mtcars = get_rdataset("mtcars", "datasets").data
mtcars.index.name = "car"
print(mtcars.head())

def get_mode(series):
    modes = series.mode(dropna=True)
    return modes.iloc[0] if not modes.empty else "Unknown"

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 1 - DATAFRAME OPERATIONS
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 1 - DATAFRAME OPERATIONS")
print("=" * 75)

dataframe1 = pd.DataFrame({
    "ProductID": range(101, 111),
    "NameofProd": ["Laptop", "Mouse", "Keyboard", "Monitor", "Printer",
                   "Scanner", "Tablet", "Webcam", "Speaker", "Headphone"],
    "Price": [65000, 800, 1500, 15000, 12000, 9000, 30000, 2500, 4000, 3500]
})

dataframe2 = pd.DataFrame({
    "ProductID": range(101, 111),
    "Company_Location": ["Chennai", "Bengaluru", "Mumbai", "Delhi", "Hyderabad",
                         "Pune", "Kolkata", "Kochi", "Coimbatore", "Jaipur"]
})

dataframe3 = pd.DataFrame({
    "ProductID": range(111, 121),
    "NameofProd": ["Smartphone", "Charger", "PowerBank", "Pendrive", "HardDisk",
                   "Microphone", "Router", "Projector", "Smartwatch", "Camera"],
    "Price": [45000, 1200, 2000, 700, 5500, 3000, 2500, 40000, 9000, 55000]
})

print("\nDataframe 1:\n", dataframe1)
print("\nDataframe 2:\n", dataframe2)
print("\nDataframe 3:\n", dataframe3)

appended_dataframe = pd.concat([dataframe1, dataframe3], ignore_index=True)
print("\nAppended Dataframe:\n", appended_dataframe)

concatenated_dataframe = pd.concat(
    [dataframe1.reset_index(drop=True), dataframe2["Company_Location"]], axis=1
)
print("\nConcatenated Dataframe:\n", concatenated_dataframe)

print("\nMerged Dataframe:\n", pd.merge(dataframe1, dataframe2, on="ProductID"))
print("\nInner Join:\n", pd.merge(dataframe1, dataframe2, on="ProductID", how="inner"))
print("\nLeft Join:\n", pd.merge(dataframe1, dataframe2, on="ProductID", how="left"))
print("\nRight Join:\n", pd.merge(dataframe1, dataframe2, on="ProductID", how="right"))
print("\nFull Outer Join:\n", pd.merge(dataframe1, dataframe2, on="ProductID", how="outer"))

# ---------------------------------------------------------------------------
# ASSIGNMENT 1 - DATA TRANSFORMATIONS
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("DATA TRANSFORMATIONS")
print("=" * 75)

Shreya = np.array([281, 45, 78, 120, 36, 95, 150, 64, 210, 18], dtype=float)
minimum_value = Shreya.min()
maximum_value = Shreya.max()

normalized_minus3_plus3 = -3 + ((Shreya - minimum_value) / (maximum_value - minimum_value)) * 6
normalized_minus1_plus1 = -1 + ((Shreya - minimum_value) / (maximum_value - minimum_value)) * 2
min_max_normalization = (Shreya - minimum_value) / (maximum_value - minimum_value)
z_score_transformation = (Shreya - Shreya.mean()) / Shreya.std(ddof=1)
decimal_places = len(str(int(np.floor(np.max(np.abs(Shreya))))))
decimal_scaling_normalization = Shreya / (10 ** decimal_places)

transformation_results = pd.DataFrame({
    "Original": Shreya,
    "Minus3_to_Plus3": normalized_minus3_plus3,
    "Minus1_to_Plus1": normalized_minus1_plus1,
    "Min_Max_0_to_1": min_max_normalization,
    "Z_Score": z_score_transformation,
    "Decimal_Scaling": decimal_scaling_normalization
})
print(transformation_results.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 2 - PRELIMINARY DATA EXPLORATION AND DATA CLEANING
# Uses the same CreditCard.csv URL.
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 2 - PRELIMINARY DATA EXPLORATION AND DATA CLEANING")
print("=" * 75)

data = pd.read_csv(CREDITCARD_URL)
print("\nFirst 10 records:\n", data.head(10))
print("\nLast 10 records:\n", data.tail(10))
print("\nRows and columns:", data.shape)
print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))
print("\nColumn names:\n", data.columns.tolist())
print("\nData types:\n", data.dtypes)
print("\nSummary statistics:\n", data.describe(include="all"))
print("\nMissing values in each column:\n", data.isna().sum())
print("\nTotal missing values:", data.isna().sum().sum())
print("\nRows with missing values:\n", data[data.isna().any(axis=1)])

clean_data = data.dropna()
print("\nDataset after removing missing values:\n", clean_data.head())

# Demonstrate mean, median and mode imputation.
imputed_data = data.copy()
numeric_cols = imputed_data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = imputed_data.select_dtypes(exclude=np.number).columns.tolist()

if len(numeric_cols) >= 1:
    col1 = numeric_cols[0]
    imputed_data[col1] = imputed_data[col1].fillna(imputed_data[col1].mean())
if len(numeric_cols) >= 2:
    col2 = numeric_cols[1]
    imputed_data[col2] = imputed_data[col2].fillna(imputed_data[col2].median())

if len(categorical_cols) >= 1:
    cat1 = categorical_cols[0]
    imputed_data[cat1] = imputed_data[cat1].fillna(get_mode(imputed_data[cat1]))

print("\nMean/median/mode imputation completed.")
forward_fill = data.ffill()
backward_fill = data.bfill()
print("\nForward fill preview:\n", forward_fill.head())
print("\nBackward fill preview:\n", backward_fill.head())

remaining_filled = data.copy()
for col in remaining_filled.columns:
    if pd.api.types.is_numeric_dtype(remaining_filled[col]):
        remaining_filled[col] = remaining_filled[col].fillna(0)
    else:
        remaining_filled[col] = remaining_filled[col].fillna("Unknown")

print("\nRemaining missing values after replacement:\n", remaining_filled.isna().sum())
print("\nDuplicate rows:", data.duplicated().sum())
data_unique = data.drop_duplicates().copy()
print("Shape after removing duplicates:", data_unique.shape)

for col in categorical_cols:
    data_unique[col] = data_unique[col].astype("category")

if len(data_unique.columns) > 0:
    old_name = data_unique.columns[0]
    data_unique = data_unique.rename(columns={old_name: old_name + "_New"})

if len(numeric_cols) >= 2:
    data_unique["Total_Value"] = (
        pd.to_numeric(data_unique[numeric_cols[0]], errors="coerce") +
        pd.to_numeric(data_unique[numeric_cols[1]], errors="coerce")
    )

print("\nCleaned data summary:\n", data_unique.describe(include="all"))

if numeric_cols:
    first_numeric = pd.to_numeric(data[numeric_cols[0]], errors="coerce").dropna()
    print("\nFirst numeric column:", numeric_cols[0])
    print("Minimum:", first_numeric.min())
    print("Maximum:", first_numeric.max())
    print("Range:", first_numeric.max() - first_numeric.min())
    print("Variance:", first_numeric.var())
    print("Standard deviation:", first_numeric.std())
    print("IQR:", first_numeric.quantile(0.75) - first_numeric.quantile(0.25))

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 3 - DATA CLEANING AND UNIVARIATE ANALYSIS
# The R document uses the columns income and owner from CreditCard.csv.
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 3 - DATA CLEANING AND UNIVARIATE ANALYSIS")
print("=" * 75)

data = pd.read_csv(CREDITCARD_URL)
print("\nFirst six records:\n", data.head(6))
print("\nLast six records:\n", data.tail(6))
print("\nTotal missing values:", data.isna().sum().sum())
print("\nRows containing missing values:\n", data[data.isna().any(axis=1)])

missing_percent = data.isna().mean() * 100
print("\nMissing percentage by column:\n", missing_percent)
data = data.loc[:, missing_percent <= 70].copy()

data_mean = data.copy()
for col in data_mean.select_dtypes(include=np.number).columns:
    data_mean[col] = data_mean[col].fillna(data_mean[col].mean())

data_median = data.copy()
for col in data_median.select_dtypes(include=np.number).columns:
    data_median[col] = data_median[col].fillna(data_median[col].median())

categorical_cols = data.select_dtypes(exclude=np.number).columns
data_mode = data.copy()
for col in categorical_cols:
    data_mode[col] = data_mode[col].fillna(get_mode(data_mode[col]))

print("\nMean-imputed data preview:\n", data_mean.head())
print("\nMedian-imputed data preview:\n", data_median.head())
print("\nMode-imputed data preview:\n", data_mode.head())

duplicates = data[data.duplicated()]
print("\nDuplicate rows:\n", duplicates)
data = data.drop_duplicates().copy()
print("\nData types:\n", data.dtypes)

char_cols = data.select_dtypes(include="object").columns
for col in char_cols:
    data[col] = data[col].astype("category")
print("\nData types after converting text columns:\n", data.dtypes)

data_lower = data.copy()
data_upper = data.copy()
for col in data.select_dtypes(include="category").columns:
    data_lower[col] = data[col].astype(str).str.lower().astype("category")
    data_upper[col] = data[col].astype(str).str.upper().astype("category")

if "income" in data.columns:
    income = pd.to_numeric(data["income"], errors="coerce").dropna()
    plt.figure()
    plt.boxplot(income)
    plt.title("Boxplot of Income")
    plt.ylabel("Income")
    plt.show()

    q1, q3 = income.quantile(0.25), income.quantile(0.75)
    iqr_value = q3 - q1
    lower_limit = q1 - 1.5 * iqr_value
    upper_limit = q3 + 1.5 * iqr_value
    outliers = data[(data["income"] < lower_limit) | (data["income"] > upper_limit)]
    print("\nIncome outliers:\n", outliers)
    data_no_outliers = data[(data["income"] >= lower_limit) & (data["income"] <= upper_limit)].copy()

    plt.figure()
    plt.hist(income, bins=20, edgecolor="black")
    plt.title("Histogram of Income")
    plt.xlabel("Income")
    plt.ylabel("Frequency")
    plt.show()

    plt.figure()
    sns.histplot(income, bins=20, stat="density", kde=True, color="lightgreen")
    plt.title("Histogram with Density Curve")
    plt.show()

    if "owner" in data.columns:
        plt.figure()
        sns.violinplot(data=data, x="owner", y="income")
        plt.title("Income by Owner")
        plt.show()

    plt.figure()
    plt.plot(income.values, "o")
    plt.title("Dot Plot of Income")
    plt.xlabel("Observation")
    plt.ylabel("Income")
    plt.show()

if "owner" in data.columns:
    freq = data["owner"].value_counts(dropna=False)
    print("\nOwner frequency table:\n", freq)
    print("\nOwner proportions:\n", data["owner"].value_counts(normalize=True))
    print("\nOwner percentages:\n", data["owner"].value_counts(normalize=True) * 100)

    freq.plot(kind="bar", title="Owner Frequency", color="steelblue")
    plt.xlabel("Owner")
    plt.ylabel("Frequency")
    plt.show()

    freq.plot(kind="barh", title="Horizontal Bar Chart", color="orange")
    plt.xlabel("Frequency")
    plt.show()

    (data["owner"].value_counts(normalize=True) * 100).plot(kind="bar", title="Percentage Bar Chart")
    plt.ylabel("Percentage")
    plt.show()

    freq.plot(kind="pie", autopct="%1.1f%%", title="Pie Chart", ylabel="")
    plt.show()

    plt.figure()
    plt.stem(range(len(freq)), freq.values)
    plt.xticks(range(len(freq)), freq.index.astype(str), rotation=45)
    plt.title("Lollipop Chart")
    plt.xlabel("Category")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(freq.values, "o")
    plt.xticks(range(len(freq)), freq.index.astype(str))
    plt.title("Category Dot Chart")
    plt.show()

if "income" in data.columns:
    plt.figure()
    sns.kdeplot(data=pd.to_numeric(data["income"], errors="coerce").dropna(), fill=True)
    plt.title("Density Plot of Income")
    plt.show()

    if "owner" in data.columns:
        plt.figure()
        sns.histplot(data=data, x="income", hue="owner", bins=20, element="step")
        plt.title("Income Distribution Across Owner Groups")
        plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# MULTIVARIATE ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("MULTIVARIATE ANALYSIS")
print("=" * 75)

data = pd.read_csv(CREDITCARD_URL)
print(data.head())
print("\nDataset types:\n", data.dtypes)

required = ["age", "income", "dependents", "owner", "active", "share", "expenditure"]
missing_required = [col for col in required if col not in data.columns]
if missing_required:
    print("Some expected columns are not present:", missing_required)
else:
    plt.figure()
    sns.scatterplot(data=data, x="age", y="income", hue="dependents")
    plt.title("Age vs Income (Color = Dependents)")
    plt.show()

    plt.figure()
    sns.scatterplot(data=data, x="age", y="income", hue="dependents", style="owner")
    plt.title("Age vs Income (Color = Dependents, Shape = Owner)")
    plt.show()

    plt.figure()
    sns.scatterplot(data=data, x="age", y="income", hue="dependents", size="active", sizes=(20, 200))
    plt.title("Age vs Income with Dependents and Active Status")
    plt.show()

    plt.figure()
    sns.scatterplot(data=data, x="age", y="income", size="dependents", sizes=(20, 200), alpha=0.5)
    plt.title("Bubble Plot")
    plt.show()

    plt.figure()
    sns.scatterplot(data=data, x="age", y="income")
    plt.title("Age vs Income")
    plt.show()

    g = sns.FacetGrid(data, col="owner", col_wrap=3)
    g.map_dataframe(sns.scatterplot, x="age", y="income", alpha=0.6)
    g.fig.suptitle("Age vs Income by Owner", y=1.03)
    plt.show()

    plt.figure()
    sns.scatterplot(data=data, x="age", y="income", hue="owner", alpha=0.6)
    plt.title("Age vs Income Colored by Owner")
    plt.show()

    pair_columns = ["age", "income", "share", "expenditure", "dependents"]
    sns.pairplot(data[pair_columns].dropna())
    plt.suptitle("Pairs Plot", y=1.02)
    plt.show()

    parallel_columns = ["age", "income", "expenditure", "dependents", "active"]
    parallel_data = data[parallel_columns].dropna().copy()
    scaled_parallel = pd.DataFrame(
        MinMaxScaler().fit_transform(parallel_data),
        columns=parallel_columns,
        index=parallel_data.index
    )
    plt.figure(figsize=(12, 6))
    for _, row in scaled_parallel.iterrows():
        plt.plot(range(len(parallel_columns)), row.values, alpha=0.15)
    plt.xticks(range(len(parallel_columns)), parallel_columns)
    plt.title("Parallel Coordinate Plot (scaled 0 to 1)")
    plt.ylabel("Scaled Value")
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 6 - TIME SERIES ANALYSIS: AirPassengers
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 6 - TIME SERIES ANALYSIS")
print("=" * 75)

print("\nFirst five observations:\n", air_passengers.head())
print("\nLast five observations:\n", air_passengers.tail())
print("Start:", air_passengers.index.min())
print("End:", air_passengers.index.max())
print("Frequency: monthly (12 observations per year)")
print("Number of observations:", len(air_passengers))
print("\nSummary:\n", air_passengers.describe())

plt.figure(figsize=(12, 5))
plt.plot(air_passengers)
plt.title("Monthly Air Passenger Trend")
plt.xlabel("Year")
plt.ylabel("Number of Passengers")
plt.show()

decomposed = seasonal_decompose(air_passengers, model="multiplicative", period=12)
decomposed.plot()
plt.suptitle("AirPassengers Decomposition")
plt.tight_layout()
plt.show()

plot_acf(air_passengers, lags=40)
plt.title("ACF of AirPassengers")
plt.show()

plot_pacf(air_passengers, lags=40, method="ywm")
plt.title("PACF of AirPassengers")
plt.show()

arima_model = SARIMAX(
    air_passengers,
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
arima_result = arima_model.fit(disp=False)
print("\nARIMA/SARIMA model summary:\n")
print(arima_result.summary())

forecast_horizon = 12
forecast_result = arima_result.get_forecast(steps=forecast_horizon)
forecast_values = forecast_result.predicted_mean
confidence_intervals = forecast_result.conf_int()
print("\nForecast for next 12 months:\n", forecast_values)

plt.figure(figsize=(12, 5))
plt.plot(air_passengers, label="Actual Data")
plt.plot(forecast_values, label="ARIMA Forecast", linestyle="--")
plt.fill_between(
    confidence_intervals.index,
    confidence_intervals.iloc[:, 0],
    confidence_intervals.iloc[:, 1],
    alpha=0.2,
    label="95% Confidence Interval"
)
plt.title("AirPassengers: Actual and Forecasted Passengers")
plt.xlabel("Year")
plt.ylabel("Number of Passengers")
plt.legend()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 7 - 1D, 2D AND ND ANALYSIS
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 7 - 1D, 2D AND ND ANALYSIS")
print("=" * 75)

df = pd.read_csv(CREDITCARD_URL)
num_var = pd.to_numeric(df["income"], errors="coerce").dropna()

stats_table = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Minimum", "Maximum", "Range",
                  "Variance", "Standard Deviation", "IQR", "Skewness", "Kurtosis"],
    "Value": [
        num_var.mean(), num_var.median(), num_var.min(), num_var.max(),
        num_var.max() - num_var.min(), num_var.var(), num_var.std(),
        num_var.quantile(0.75) - num_var.quantile(0.25),
        stats.skew(num_var, bias=False), stats.kurtosis(num_var, bias=False)
    ]
})
print("\nSummary statistics:\n", stats_table)

freq_table = df["owner"].value_counts()
rel_freq_table = df["owner"].value_counts(normalize=True)
print("\nFrequency table:\n", freq_table)
print("\nRelative frequency table:\n", rel_freq_table)

freq_table.plot(kind="pie", autopct="%1.1f%%", title="Pie Chart of Owner Status", ylabel="")
plt.show()
freq_table.plot(kind="bar", title="Bar Plot of Owner Status", color="skyblue")
plt.xlabel("Owner")
plt.ylabel("Count")
plt.show()

age_bins = pd.cut(df["age"], bins=[20, 30, 40, 50, 60], include_lowest=True)
print("\nAge frequency distribution:\n", age_bins.value_counts().sort_index())

plt.figure()
plt.hist(df["age"].dropna(), bins=5, edgecolor="black")
plt.title("Histogram of Age")
plt.xlabel("Age")
plt.show()

plt.figure()
plt.boxplot(df["age"].dropna())
plt.title("Box Plot of Age")
plt.ylabel("Age")
plt.show()

contingency_table = pd.crosstab(df["card"], df["owner"])
print("\nCross-tabulation (Card vs Owner):\n", contingency_table)
print("\nRow percentages:\n", pd.crosstab(df["card"], df["owner"], normalize="index") * 100)
print("\nColumn percentages:\n", pd.crosstab(df["card"], df["owner"], normalize="columns") * 100)

contingency_table.plot(kind="bar", figsize=(8, 5))
plt.title("Grouped Bar Plot: Card Status by Ownership")
plt.xlabel("Card")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

grouped_stats = df.groupby("card")["income"].agg(["mean", "median", "std"])
print("\nIncome grouped by card:\n", grouped_stats)

plt.figure()
sns.violinplot(data=df, x="card", y="income", inner=None)
sns.boxplot(data=df, x="card", y="income", width=0.25, color="white")
plt.title("Side-by-Side Box and Violin Plot")
plt.xlabel("Card Approved")
plt.ylabel("Income")
plt.show()

pearson_corr = df["income"].corr(df["expenditure"], method="pearson")
spearman_corr = df["income"].corr(df["expenditure"], method="spearman")
print("\nPearson Correlation (Income & Expenditure):", pearson_corr)
print("Spearman Correlation (Income & Expenditure):", spearman_corr)

plt.figure()
sns.regplot(data=df, x="income", y="expenditure", scatter_kws={"alpha": 0.5})
plt.title("Scatter Plot with Linear Regression Line")
plt.show()

agg_2cat = df.groupby(["card", "owner"])["income"].agg(["mean", "std"]).reset_index()
agg_2cat["std"] = agg_2cat["std"].fillna(0)
print("\nMean and standard deviation by Card and Owner:\n", agg_2cat)

g = sns.catplot(data=df, x="card", y="income", col="owner", kind="box")
g.fig.suptitle("Faceted Box Plot", y=1.03)
plt.show()

plt.figure()
sns.barplot(data=df, x="card", y="income", hue="owner", errorbar="sd")
plt.title("Grouped Bar Chart with Error Bars")
plt.ylabel("Mean Income")
plt.show()

plt.figure()
sns.scatterplot(data=df, x="income", y="expenditure", hue="card", style="card")
plt.title("Scatter Plot Stratified by Card Approval")
plt.show()

print("\nCorrelation between income and share for each card category:")
for category, group in df.groupby("card"):
    print(category, group[["income", "share"]].corr())

num_vars = ["age", "income", "share", "expenditure"]
cor_matrix = df[num_vars].corr(method="pearson")
print("\nCorrelation matrix:\n", cor_matrix)

plt.figure(figsize=(7, 5))
sns.heatmap(cor_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix Heatmap")
plt.show()

sns.pairplot(df[num_vars].dropna())
plt.suptitle("Pairs Plot Matrix", y=1.02)
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# ASSIGNMENT 8 - K-MEANS AND HIERARCHICAL CLUSTERING
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("ASSIGNMENT 8 - K-MEANS CLUSTERING")
print("=" * 75)

cluster_columns = ["reports", "age", "income", "share", "expenditure",
                   "dependents", "months", "majorcards", "active"]
cluster_data = df[cluster_columns].apply(pd.to_numeric, errors="coerce")
cluster_data = cluster_data.fillna(cluster_data.median())
scaled_data = StandardScaler().fit_transform(cluster_data)

print("\nDependent variable distribution (card):\n", df["card"].value_counts())

kmeans2 = KMeans(n_clusters=2, random_state=123, n_init=10)
kmeans2_labels = kmeans2.fit_predict(scaled_data)
print("\nK-Means (k=2) cluster centers:\n", kmeans2.cluster_centers_)
print("Cluster counts:\n", pd.Series(kmeans2_labels).value_counts().sort_index())

wcss = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=123, n_init=10)
    model.fit(scaled_data)
    wcss.append(model.inertia_)

print("\nWCSS for k=1 to 10:\n", wcss)
plt.figure()
plt.plot(range(1, 11), wcss, marker="o")
plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Plot")
plt.xticks(range(1, 11))
plt.show()

# The original R assignment selected 3 clusters after the elbow plot.
final_kmeans = KMeans(n_clusters=3, random_state=123, n_init=10)
final_clusters = final_kmeans.fit_predict(scaled_data)
print("\nFinal K-Means cluster counts:\n", pd.Series(final_clusters).value_counts().sort_index())

print("\n" + "=" * 75)
print("HIERARCHICAL CLUSTERING")
print("=" * 75)

# Ward hierarchical clustering is computationally expensive on the full dataset.
# The original dataset is retained; use all rows when feasible. For very large
# datasets, this section uses a reproducible sample to keep Colab responsive.
if len(scaled_data) > 2000:
    rng = np.random.default_rng(123)
    sample_indices = rng.choice(len(scaled_data), size=2000, replace=False)
    hierarchical_data = scaled_data[sample_indices]
    print("Using a reproducible sample of 2000 rows for hierarchical clustering.")
else:
    hierarchical_data = scaled_data

distance_linkage = linkage(hierarchical_data, method="ward", metric="euclidean")
plt.figure(figsize=(14, 6))
dendrogram(distance_linkage, truncate_mode="lastp", p=30)
plt.title("Hierarchical Clustering Dendrogram (Ward)")
plt.xlabel("Samples / merged clusters")
plt.ylabel("Distance")
plt.show()

hierarchical_clusters = fcluster(distance_linkage, t=3, criterion="maxclust")
print("\nHierarchical cluster assignments (first 30):\n", hierarchical_clusters[:30])
print("\nNumber of samples in each hierarchical cluster:\n",
      pd.Series(hierarchical_clusters).value_counts().sort_index())

# ---------------------------------------------------------------------------
# ASSIGNMENT 8 - PCA USING mtcars
# ---------------------------------------------------------------------------

print("\n" + "=" * 75)
print("PCA - mtcars DATASET")
print("=" * 75)

cars_pca = mtcars[["mpg", "disp", "hp", "drat", "wt", "qsec"]].copy()
print("\nSelected mtcars variables:\n", cars_pca.head())
print("\nSummary statistics:\n", cars_pca.describe())

cars_scaled = StandardScaler().fit_transform(cars_pca)
pca_result = PCA()
pca_scores = pca_result.fit_transform(cars_scaled)

eigenvalues = pca_result.explained_variance_
variance_proportion = pca_result.explained_variance_ratio_
variance_percent = variance_proportion * 100
cumulative_percent = np.cumsum(variance_percent)

pca_summary = pd.DataFrame({
    "Principal_Component": [f"PC{i}" for i in range(1, len(eigenvalues) + 1)],
    "Eigenvalue": eigenvalues,
    "Variance_Percent": variance_percent,
    "Cumulative_Percent": cumulative_percent
})
print("\nPCA variance summary:\n", pca_summary)

num_pc = np.argmax(cumulative_percent >= 80) + 1
print("\nMinimum number of PCs required for at least 80% variance:", num_pc)
print("Cumulative variance explained:", cumulative_percent[num_pc - 1], "%")

plt.figure()
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")
plt.axhline(y=1, linestyle="--")
plt.title("Scree Plot - PCA of mtcars")
plt.xlabel("Principal Component")
plt.ylabel("Eigenvalue")
plt.xticks(range(1, len(eigenvalues) + 1))
plt.show()

loadings = pd.DataFrame(
    pca_result.components_.T,
    index=cars_pca.columns,
    columns=[f"PC{i}" for i in range(1, len(eigenvalues) + 1)]
)
print("\nPCA loadings:\n", loadings)
print("\nLargest absolute loadings for PC1:\n", loadings["PC1"].abs().sort_values(ascending=False))
print("\nLargest absolute loadings for PC2:\n", loadings["PC2"].abs().sort_values(ascending=False))

# PCA biplot
plt.figure(figsize=(10, 7))
plt.scatter(pca_scores[:, 0], pca_scores[:, 1])
for i, car in enumerate(cars_pca.index):
    plt.text(pca_scores[i, 0], pca_scores[i, 1], str(car), fontsize=8)

scale_factor = np.max(np.abs(pca_scores[:, :2])) * 0.7
for i, variable in enumerate(cars_pca.columns):
    plt.arrow(
        0, 0,
        pca_result.components_[0, i] * scale_factor,
        pca_result.components_[1, i] * scale_factor,
        color="red",
        head_width=0.08,
        length_includes_head=True
    )
    plt.text(
        pca_result.components_[0, i] * scale_factor * 1.12,
        pca_result.components_[1, i] * scale_factor * 1.12,
        variable,
        color="red"
    )
plt.axhline(0, linestyle="--", linewidth=0.7)
plt.axvline(0, linestyle="--", linewidth=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Biplot - mtcars")
plt.tight_layout()
plt.show()

print("\n" + "=" * 75)
print("ALL ASSIGNMENT SECTIONS FINISHED")
print("=" * 75)